# Human-Swarm Interaction with SHaSTA

SHaSTA simulates **UAV and UGV swarms on real city maps**. A human operator, a script, or a learning agent commands groups of vehicles, and the
operator's orders can be recorded on one clock with eye gaze. In this notebook **you play the operator, right here in the browser**, while you learn
what is inside it: the map and the config file, the Gym-style environment, the interface and the commander behind it, how to write your own
experiment, and how a session is recorded with Lab Streaming Layer (LSL) and analysed.

## Overview

| Step | What you do |
|---|---|
| 1. The map and the config file | Load a real city as a street graph, and read the simulation's settings from `config.yaml` |
| 2. Actors, groups and the Gym API | Build a swarm, and order a group to a node with `env.step`, with no human |
| 3. Human in the loop | Play the operator in 2D and in 3D: click a group, click a node, send it. The commander behind the interface |
| 4. What you can customize | Change the swarm, the targets and the time limit, and play your own mission |
| 5. Write your own experiment | Subclass `GoToNodeExperiment`: your own rules, reward and done condition |
| 6. LSL basics | Stream and read time-stamped data with `pylsl` |
| 7. Record the operator and gaze | Play again while your orders and (synthetic) eye gaze are recorded to an XDF file |
| 8. Analyse the recording | Load the file and see how the gaze relates to your orders |

## How to play

Wherever a cell opens the interface, **click the picture first**, then use the mouse and the keys. Press **Stop** to end and see your score.

| Do this | To |
|---|---|
| Click a group's marker, or press `1`-`9` | pick a group |
| Click near a street node | pick a target (a white cross marks it) |
| `Enter`, or the **Send order** button | send the group to the target |
| `Shift` + click, or `Shift` + drag | select several groups |
| Mouse wheel, right-drag | zoom, pan (`R` resets the view) |
| `Space`, `+` / `-` | pause, speed |
| `V` | the 3D camera view of the selected group |

## Before you start

Select the **Python (smc)** kernel. Keep `config.yaml`, `lsl_tools.py`, `live_play.py` and `world_view.py` in the same folder as this notebook.

In [ ]:
# Setup: draw pygame off-screen and keep the output quiet. This must run before pygame is imported.
import os, sys, tempfile, warnings, logging, contextlib, ctypes
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"        # no pygame banner
warnings.filterwarnings("ignore")                     # no warnings
os.environ["PYTHONWARNINGS"] = "ignore"               # ... also in anything this notebook starts
logging.getLogger().setLevel(logging.ERROR)           # ... and no log messages below errors
with tempfile.NamedTemporaryFile("w", suffix=".cfg", delete=False) as lsl_config:
    lsl_config.write("[log]\nlevel = -2\n")            # no log lines from the LSL library
os.environ["LSLAPICFG"] = lsl_config.name
sys.path.insert(0, os.getcwd())

@contextlib.contextmanager
def quiet_c_output():
    """PyBullet prints from C code, which Python's warnings and logging do not reach: silence it while this block runs."""
    sys.stdout.flush(); sys.stderr.flush()
    saved, devnull = (os.dup(1), os.dup(2)), os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 1); os.dup2(devnull, 2)
    try:
        yield
    finally:
        ctypes.CDLL(None).fflush(None)                  # C buffers its output: flush it before the real streams come back
        os.dup2(saved[0], 1); os.dup2(saved[1], 2)
        for fd in (*saved, devnull):
            os.close(fd)

with quiet_c_output():
    import pybullet
    from shasta.env import ShastaEnv

def _quiet(method):
    def wrapper(*args, **kwargs):
        with quiet_c_output():
            return method(*args, **kwargs)
    return wrapper

for name in ("__init__", "reset", "close"):             # PyBullet prints when an environment connects, resets and closes
    setattr(ShastaEnv, name, _quiet(getattr(ShastaEnv, name)))

## Step 1: The map and the config file

SHaSTA builds its world from **OpenStreetMap** data. A map holds the street network, the buildings, and a table that converts latitude and longitude
to the simulator's metre coordinates. The `Map` class works on its own, with no physics.

Roads are stored as a **graph**: *nodes* are intersections and *edges* are the streets between them. Everything we do later, such as ordering a
swarm to go somewhere, is expressed in these nodes.

In [ ]:
from shasta.map import Map

world_map = Map()
world_map.setup({"map_to_use": "buffalo-small"})
graph = world_map.get_node_graph()
print("intersections (nodes):", graph.number_of_nodes(), "  street segments (edges):", graph.number_of_edges())

node = 5 if 5 in graph else next(iter(graph.nodes))
print(f"node {node} is at", world_map.get_cartesian_node_position(node).round(1), "metres, latitude/longitude", world_map.get_lat_lon_node_position(node))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

fig, ax = plt.subplots(figsize=(7, 7))
for outline in world_map.get_building_footprints():
    ax.add_patch(Polygon(outline, closed=True, facecolor="#c9ccd3", edgecolor="#8d93a1", linewidth=0.6))
for segment in world_map.get_street_segments():
    ax.plot(segment[:, 0], segment[:, 1], color="#3b6fb6", linewidth=1.2)
for n in graph.nodes:
    x, y = world_map.get_cartesian_node_position(n)[:2]
    ax.plot(x, y, "o", color="white", markeredgecolor="black", markersize=9)
    ax.text(x, y, str(n), ha="center", va="center", fontsize=6)
ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.set_title("buffalo-small: buildings (grey), streets (blue), intersections (numbered)")
plt.show()

A simulation is described by a **config file**: a short YAML text file next to this notebook. The notebook reads `config.yaml` and builds everything from it,
so changing a number there changes the simulation. Here it is:

In [ ]:
from shasta.config import load_config

CONFIG_PATH = "config.yaml"
print(open(CONFIG_PATH).read())
config = load_config(CONFIG_PATH)

Which part is read by whom:

| Keys | Read by | Meaning |
|---|---|---|
| `headless`, `time_step`, `experiment.map_to_use` | SHaSTA | window or not, physics step, which map |
| `swarm` | this notebook | how many vehicles, where they start, how fast they are |
| `mission` | this notebook | targets, reach radius, time limit |
| `human`, `collector` | this notebook | the operator's interface, the experiment of Step 5 |

## Step 2: Actors, groups and the Gym API

An **actor** is one vehicle: a `UAV` (a drone, which flies) or a `UGV` (a ground vehicle, which drives). Actors are organised in **groups**: a dictionary
`{group_id: [actors]}`. A group is what an operator commands.

The **environment** (`ShastaEnv`) combines the map, the PyBullet physics world and an *experiment*, which defines what the actors do and what we observe.
We use `GoToNodeExperiment`: "go to a map node, in formation". It follows the Gymnasium API, like any reinforcement-learning environment.

In [ ]:
import copy, time
from shasta.actors import UAV, UGV
from shasta.experiments import GoToNodeExperiment

def vehicle_group(kind, n, node, speed):
    """``n`` vehicles of type ``kind`` in a small ring around map node ``node``, each limited to ``speed`` m/s."""
    lat, lon = world_map.get_lat_lon_node_position(node)
    vehicles = []
    for angle in (np.linspace(0, 2 * np.pi, n, endpoint=False) if n > 1 else [0.0]):
        ring = 5e-5 if n > 1 else 0.0                                   # about 5 m, in degrees
        vehicle = kind(init_pos=[lat + ring * np.sin(angle), lon + ring * np.cos(angle)])
        vehicle.speed = speed
        vehicles.append(vehicle)
    return vehicles

def build_groups(config, uavs=None, ugvs=None):
    """The actor groups described by ``config['swarm']``: group 0 = UAVs, group 1 = UGVs."""
    s = config["swarm"]
    return {0: vehicle_group(UAV, uavs or s["uavs"], s["uav_start_node"], s["uav_speed"]),
            1: vehicle_group(UGV, ugvs or s["ugvs"], s["ugv_start_node"], s["ugv_speed"])}

def make_env(groups, config=config, experiment=GoToNodeExperiment, experiment_config=None):
    """Build a ShastaEnv from a config dictionary."""
    config = copy.deepcopy(config)
    config["experiment"] = {**config.get("experiment", {}), "type": experiment}
    if experiment_config is not None:
        config["experiment"]["config"] = experiment_config
    return ShastaEnv(config, groups)

env = make_env(build_groups(config, uavs=1, ugvs=1))
obs, info = env.reset()                                    # the Gym API: reset() starts an episode
for group_id, positions in obs.items():
    print(f"group {group_id}: positions (x, y, z) in metres ->", positions.round(1).tolist())

The **observation** is a dictionary `{group_id: array of shape (n_actors, 3)}`: the position of every vehicle. Because the config fixes the start nodes,
the vehicles start in the same place on every run.

### Order a group, with no human

`env.step(action)` advances the simulation and returns `(observation, reward, terminated, truncated, info)`. Here the **action is an order**:
`{group_id: node}`; `env.step(None)` means "keep going". Nobody is at the keyboard: a script, or an agent trained with reinforcement learning,
uses these same calls, with its own policy choosing the node.

Under the hood the physics is **PyBullet**, run in `DIRECT` mode: no window and no graphics card. A mission runs as fast as the CPU can compute it.

In [ ]:
def farthest_node(world_map, xy):
    return max(graph.nodes, key=lambda n: np.linalg.norm(world_map.get_cartesian_node_position(n)[:2] - xy))

start = obs[0].mean(axis=0)[:2]
target = farthest_node(world_map, start)
started = time.time()

obs, reward, terminated, truncated, info = env.step({0: target})      # the order: send the UAV to the farthest node
steps = 1
while not terminated and steps < 20000:
    obs, reward, terminated, truncated, info = env.step(None)         # keep going
    steps += 1
travelled = np.linalg.norm(obs[0].mean(axis=0)[:2] - start)
print(f"ordered group 0 to node {target}: terminated={terminated} after {steps} steps, {travelled:.0f} m from where it started")
print(f"the whole trip took {time.time() - started:.1f} s of computing, with no window")
env.close()

## Step 3: Human in the loop

`SwarmCommander` is the layer a human, an interface, or an AI advisor talks to. It can **select** groups and **order** them to nodes
(`commander.send(group, node)`), **advance** the simulation (`commander.update()`), and report **status** (position, battery, distance to target, ETA)
and an **event log**. A **Mission** is a list of target nodes; it is complete when every target has been reached by some group.

`ShastaGUI` is the pygame window built on top of exactly this commander. Run the cell, click the picture, and **command the swarm**: pick a group, pick a
target node, send it, and watch the groups follow the streets. The mission targets are the yellow markers.

In [ ]:
from shasta.gui import ShastaGUI
from shasta.interaction import Mission, SwarmCommander
from live_play import play_shasta

def make_mission(world_map, cfg):
    m = cfg["mission"]
    return Mission.random(world_map, n_targets=m["targets"], seed=m["seed"], radius=m["radius"], time_limit=m["time_limit"])

def make_commander(cfg=config, uavs=None, ugvs=None):
    """A commander for the swarm and the mission described by ``cfg``."""
    env = make_env(build_groups(cfg, uavs=uavs, ugvs=ugvs), config=cfg)
    env.reset()
    return SwarmCommander(env, make_mission(env.core.get_map(), cfg), steps_per_update=cfg["human"]["steps_per_update"])

commander = make_commander()
play_shasta(ShastaGUI(commander))

### Interact with the PyBullet world

PyBullet needs no window, but it can still **render the 3D scene to an image** (`getCameraImage`). That is enough to interact with it directly. The view below draws the
3D world, lets you move the camera, and turns a **click on the ground** into an order: the click becomes a ray into the world, the ray meets the ground, and the nearest street
node is sent to the selected group with `commander.send`.

| Do this | To |
|---|---|
| Left-drag, right-drag, mouse wheel | orbit, pan, zoom the camera |
| Click the ground | send the selected group to the nearest street node |
| `1`-`9` | pick a group |
| `Space`, `+` / `-`, `F`, `R` | pause, speed, follow the selected group, reset the camera |

The code is in `world_view.py`, next to the notebook: open it to see how little it takes.

In [ ]:
from world_view import play_world

play_world(make_commander())

### No human needed

The interface is only a window on `SwarmCommander`, which has no display. Anything that can call it can be the operator. Here a **scripted operator**
plays the same mission with no window: whenever a group is free, it sends it to the next target that has not been assigned.

In [ ]:
def scripted_operator(commander, pending):
    """Give every free group the next unassigned target."""
    for group_id in commander.groups:
        free = group_id not in commander.orders or commander.status(group_id)["arrived"]
        if pending and free:
            commander.send(group_id, pending.pop(0))

commander = make_commander()
pending = list(commander.mission.targets)
while not commander.mission.is_over(commander.step_count) and commander.step_count < 20000:
    scripted_operator(commander, pending)
    commander.update()
print(f"scripted operator: targets reached {commander.mission.score} of {len(commander.mission.targets)}, {commander.step_count} steps")
for step, text in commander.events:
    print(f"  step {step:5d}: {text}")
commander.env.close()

## Step 4: What you can customize

Everything about the mission is a setting. The swarm (how many vehicles, how fast), the mission (how many targets, how close counts as reaching one,
how much time), and the seed (the same seed gives the same targets, so trials are comparable). Change the numbers and run the cell to **play your own mission**:
more vehicles to supervise, more targets, less time.

In [ ]:
UAVS, UGVS = 4, 3                   # vehicles in each group
SPEED = 2.5                         # metres per second: the most a vehicle moves each step
TARGETS, RADIUS = 3, 12.0           # targets to reach, and how close counts as reached (metres)
TIME_LIMIT = 2500                   # simulation steps, or None for no limit
SEED = 0                            # the same seed gives the same targets

my_config = copy.deepcopy(config)
my_config["swarm"].update(uav_speed=SPEED, ugv_speed=SPEED)
my_config["mission"].update(targets=TARGETS, radius=RADIUS, time_limit=TIME_LIMIT, seed=SEED)

play_shasta(ShastaGUI(make_commander(my_config, uavs=UAVS, ugvs=UGVS)))

## Step 5: Write your own experiment

So far we used the ready-made `GoToNodeExperiment`. A research study needs its own rules: what the actors do, what we observe, **when the episode ends**
and **how performance is scored**. An experiment is a class that fills in four methods of `BaseExperiment`:

| Method | Question it answers |
|---|---|
| `apply_actions(actions, core)` | What do the vehicles do this tick? |
| `get_observation(observation, core)` | What does the agent or human get to see? |
| `get_done_status(observation, core)` | Is the episode over? |
| `compute_reward(observation, core)` | How well did it go this tick? |

`GoToNodeExperiment` already does the first two (route planning and flying in formation), so we **extend it** and add our own done condition and reward.
Our experiment, **TargetCollector**: *one UAV group must visit a list of nodes, one after another. It gets +1 for every node reached and −0.001 for every step.*
How it picks the **next node** is a *strategy* we pass in: a function `(centroid, remaining_nodes, world_map) -> next_node`.

In [ ]:
class TargetCollector(GoToNodeExperiment):
    """One group (id 0) visits ``exp_config['nodes']``; the next node is chosen by ``exp_config['strategy']``."""

    def reset(self):
        super().reset()
        self.targets, self.paths = {}, {}
        self.remaining = list(self.exp_config["nodes"])
        self.visited, self.current, self.arrived_now = [], None, False

    def _centroid(self, core):
        actors = core.get_actors_by_group_id(0)
        return np.mean([a.get_pos_and_orientation()[0] for a in actors], axis=0)[:2]

    def _reached(self, centroid, core):
        node_xy = core.get_map().get_cartesian_node_position(self.current)[:2]
        return np.linalg.norm(centroid - node_xy) <= self.tolerance

    def apply_actions(self, actions, core):
        self.arrived_now = False
        centroid = self._centroid(core)
        if self.current is not None and self._reached(centroid, core):
            self.visited.append(self.current); self.current, self.arrived_now = None, True
        if self.current is None and self.remaining:
            self.current = self.exp_config["strategy"](centroid, self.remaining, core.get_map())
            self.remaining.remove(self.current)
            self.set_target(0, self.current)            # plan the route (inherited)
            if len(self.paths[0]) == 0:                 # already at the start of the route: go straight there
                self.paths[0] = np.array([core.get_map().get_cartesian_node_position(self.current)])
        super().apply_actions(None, core)               # fly the route in formation (inherited)

    def get_done_status(self, observation, core):
        if self.current is not None and not self.remaining and self._reached(self._centroid(core), core):
            self.visited.append(self.current); self.current, self.arrived_now = None, True
        return self.current is None and not self.remaining and len(self.visited) > 0

    def compute_reward(self, observation, core):
        return (1.0 if self.arrived_now else 0.0) - 0.001

Two strategies on the same five nodes: **nearest** always flies to the closest remaining node, **random** visits them in a random order.

In [ ]:
import pandas as pd

def nearest(centroid, remaining, world_map):
    return min(remaining, key=lambda n: np.linalg.norm(world_map.get_cartesian_node_position(n)[:2] - centroid))

def make_random(seed):
    rng = np.random.default_rng(seed)
    return lambda centroid, remaining, world_map: remaining[int(rng.integers(len(remaining)))]

def run_episode(nodes, strategy, max_steps=60000):
    s = config["swarm"]
    group = vehicle_group(UAV, 3, s["uav_start_node"], s["uav_speed"])
    for vehicle in group:
        vehicle.battery = 1000        # the default 100 lasts about 2000 m; visiting five nodes can need more
    env = make_env({0: group}, experiment=TargetCollector, experiment_config={"nodes": nodes, "strategy": strategy})
    obs, _ = env.reset()
    trace, total_reward, done, steps = [obs[0].mean(axis=0)[:2]], 0.0, False, 0
    while not done and steps < max_steps:
        obs, reward, done, _, _ = env.step(None)
        total_reward += reward; steps += 1
        trace.append(obs[0].mean(axis=0)[:2])
    visited = list(env.experiment.visited)
    env.close()
    return {"steps": steps, "distance_m": float(np.linalg.norm(np.diff(np.array(trace), axis=0), axis=1).sum()),
            "reward": total_reward, "visit order": visited, "finished": done}

nodes = [int(n) for n in np.random.default_rng(3).choice(list(graph.nodes), config["collector"]["n_nodes"], replace=False)]
print("nodes to visit:", nodes)
rows = [{"strategy": name, **run_episode(nodes, strategy)} for name, strategy in (("nearest", nearest), ("random", make_random(0)))]
display(pd.DataFrame(rows).round(2))

## Step 6: LSL basics with `pylsl`

**Lab Streaming Layer (LSL)** is the standard way to move time-stamped data between programs and devices: eye trackers, EEG, task software.
Everyone publishes a **stream** and anyone on the network can read it, with all streams sharing **one clock**, so they can be lined up later.

- An **outlet** publishes samples. A stream has a *name*, a *type* (`Markers`, `Gaze`, `EEG`, ...), a number of *channels*, a *sampling rate*
  (`IRREGULAR_RATE` for events) and a *data format*.
- An **inlet** finds a stream and reads samples, each with an LSL **timestamp**.

Below, one program is both sides, which is the simplest way to see how it works.

In [ ]:
import time, pylsl

info = pylsl.StreamInfo(name="demo-events", type="Markers", channel_count=1,
                        nominal_srate=pylsl.IRREGULAR_RATE, channel_format="string", source_id="demo-1")
outlet = pylsl.StreamOutlet(info)

found = pylsl.resolve_byprop("name", "demo-events", timeout=5)
print("streams found:", [(s.name(), s.type(), s.channel_count()) for s in found])

inlet = pylsl.StreamInlet(found[0])
inlet.open_stream()

t0 = pylsl.local_clock()
for word in ["start", "order group 0", "order group 1", "end"]:
    outlet.push_sample([word])
    time.sleep(0.25)

samples, stamps = inlet.pull_chunk(timeout=1.0)
for sample, stamp in zip(samples, stamps):
    print(f"t = {stamp - t0:5.2f} s   {sample[0]}")
del outlet, inlet

**Checkpoint.** The events arrive with timestamps about 0.25 s apart. Every sample carries a timestamp from LSL's own clock (`pylsl.local_clock()`). That clock is what lets
us line up **orders (from SHaSTA)** and **gaze (from the eye tracker)**, even though they come from different programs.

## Step 7: Record the operator and gaze

Now we record a human-swarm session **while you play it**. We need three things:

1. A **marker stream** from SHaSTA: every order you give, with where the target is *on the screen*.
2. A **gaze stream** from an eye tracker: where the operator is looking, in screen pixels. Here it is **synthetic**: made-up gaze that looks at the target you just ordered, then follows the selected group. No hardware is needed.
3. A **recorder** that saves both into one **XDF** file, the standard LSL recording format.

Run the two cells, play for a minute (give a few orders), then press **Stop**. A real eye tracker streams in the same way: it publishes a `Gaze` stream
from the laptop it is plugged into, and the recorder picks it up. For the real study, record on that laptop and bring the XDF file here.

In [ ]:
import time
import pygame
from lsl_tools import MarkerOutlet, SyntheticGaze, Recorder, attach_markers, attach_order_markers, screen_xy

XDF_PATH = "session.xdf"

commander = make_commander()
gui = ShastaGUI(commander)
view = gui.view

# 1. Event markers from SHaSTA, with each order's target position on the screen
markers = MarkerOutlet("SHASTA-Events")
attach_markers(commander, markers)
attach_order_markers(commander, markers, view)

# 2. Gaze: synthetic here. With the real tracker this is a stream from tobii_to_lsl.py instead.
gaze = SyntheticGaze(width=gui.size[0], height=gui.size[1]).start()

# 3. The recorder: picks up exactly these two streams and writes one XDF file
own = {markers.source_id, gaze.source_id}
recorder = Recorder(XDF_PATH, stream_types=("Markers", "Gaze"), source_ids=own).start()
print("recording:", recorder.streams)

In [ ]:
markers.push("session_start", screen=list(gui.size), map_rect=list(gui.map_rect), targets=commander.mission.targets, synthetic_gaze=True)

seen, hold_until = 0, 0.0
original_step = gui.step

def step_and_look(*args, **kwargs):
    """One frame of the interface, then point the synthetic gaze: at the newest order's target, then at the selected group."""
    global seen, hold_until
    original_step(*args, **kwargs)
    if len(commander.events) != seen:                                   # a new order or arrival
        seen = len(commander.events)
        if commander.orders:
            newest = list(commander.orders.values())[-1]
            gaze.look_at(*screen_xy(view, commander.node_position(newest)))
            hold_until = time.time() + 0.8
    if time.time() >= hold_until:
        gaze.look_at(*screen_xy(view, commander.status(commander.selected)["centroid"]))
    if gui.frame == 120:
        pygame.image.save(gui.screen, "session_frame.png")             # the background of the gaze plots in Step 8

gui.step = step_and_look

def finish():
    """Runs once when the interface ends: close the recording."""
    markers.push("session_end", score=commander.mission.score, steps=commander.step_count)
    saved = recorder.stop()
    markers.close()
    gaze.stop()
    commander.env.close()
    return f"saved {saved} ({os.path.getsize(saved) / 1024:.0f} KB)"

play_shasta(gui, on_stop=finish)

## Step 8: Analyse the recording

`pyxdf.load_xdf` reads every stream in the file together with its timestamps. Our helper `load_streams` returns them by type.
Run this after you have pressed **Stop** in Step 7. To analyse a real session instead, set `XDF_PATH` to your own LabRecorder file. Nothing else changes.

In [ ]:
import pandas as pd
from lsl_tools import load_streams, gaze_table, marker_table, detect_fixations, gaze_heatmap

XDF_PATH = "session.xdf"          # <- change to your LabRecorder file for a real recording
streams = load_streams(XDF_PATH)

for stream in streams.values():
    span = stream["ts"][-1] - stream["ts"][0]
    print(f"{stream['type']:8s} {stream['name']:16s} {len(stream['ts']):6d} samples over {span:5.1f} s"
          f" ({len(stream['ts']) / span:5.1f} Hz){'   <-- SYNTHETIC data' if stream['synthetic'] else ''}")

Both streams share one clock, so we can put them on a common time axis, starting at zero. We split the markers into the session start and the orders.

In [ ]:
gaze_data = gaze_table(streams["Gaze"])
events = marker_table(streams["Markers"])

t0 = min(gaze_data["t"][0], events[0]["t"])
gt = gaze_data["t"] - t0
for e in events: e["t"] -= t0

start = next(e for e in events if e["event"] == "session_start")
SCREEN = tuple(start["screen"])                     # (width, height) in pixels
MAP_RECT = start["map_rect"]                        # x, y, w, h of the map panel
orders = [e for e in events if e["event"] == "order"]

print("screen:", SCREEN, "  orders:", len(orders), "  valid gaze samples:", f"{gaze_data['valid'].mean():.0%}")
assert orders, "This recording has no orders. In Step 7, give at least one order in the interface, then press Stop, and run this step again."
pd.DataFrame(orders)[["t", "group", "node", "screen_x", "screen_y"]].round(1)

### Gaze over time, with the operator's orders

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(gt, gaze_data["x"], label="gaze x (px)", linewidth=0.8)
ax.plot(gt, gaze_data["y"], label="gaze y (px)", linewidth=0.8)
for e in orders:
    ax.axvline(e["t"], color="red", linestyle="--", alpha=0.7)
    ax.text(e["t"], ax.get_ylim()[1], f" order g{e['group']}", color="red", va="top", fontsize=8)
ax.set_xlabel("time (s)"); ax.set_ylabel("screen position (px)"); ax.legend(loc="lower right")
plt.show()

### Where did the operator look?

In [ ]:
fixations = detect_fixations(gaze_data["t"], gaze_data["x"], gaze_data["y"], max_dispersion=40, min_duration=0.10)
print(len(fixations), "fixations, median duration", f"{np.median([f['end'] - f['start'] for f in fixations]) * 1000:.0f} ms")

heat = gaze_heatmap(gaze_data["x"], gaze_data["y"], shape=(SCREEN[1], SCREEN[0]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
background = plt.imread("session_frame.png") if os.path.exists("session_frame.png") else None
for ax in axes:
    if background is not None:
        ax.imshow(background, extent=[0, SCREEN[0], SCREEN[1], 0])
    ax.set_xlim(0, SCREEN[0]); ax.set_ylim(SCREEN[1], 0); ax.axis("off")
axes[0].imshow(heat, extent=[0, SCREEN[0], SCREEN[1], 0], cmap="hot", alpha=0.55); axes[0].set_title("Gaze heatmap")
for f in fixations:
    axes[1].scatter(f["x"], f["y"], s=(f["end"] - f["start"]) * 3000, facecolors="none", edgecolors="cyan", linewidths=1.5)
axes[1].set_title("Fixations (circle size = duration)")
plt.tight_layout(); plt.show()

### Reaction to each order

After each order, how long did the operator take to look at the target, and for how long? This only makes sense with a real tracker; the synthetic gaze is built to look at the target
right after the order, so here it shows the method.

In [ ]:
rows = []
for e in orders:
    target_px = np.array([e["screen_x"], e["screen_y"]])
    later = [f for f in fixations if f["start"] - t0 >= e["t"]]
    hit = next((f for f in later if np.linalg.norm([f["x"], f["y"]] - target_px) <= 60), None)
    rows.append({"order_t": e["t"], "group": e["group"], "node": e["node"],
                 "latency_s": (hit["start"] - t0 - e["t"]) if hit else np.nan,
                 "dwell_s": (hit["end"] - hit["start"]) if hit else np.nan})
reaction = pd.DataFrame(rows).round(2)
display(reaction)
print("mean latency to look at the target:", reaction["latency_s"].mean().round(2), "s")

on_map = ((gaze_data["x"] >= MAP_RECT[0]) & (gaze_data["x"] <= MAP_RECT[0] + MAP_RECT[2]) &
          (gaze_data["y"] >= MAP_RECT[1]) & (gaze_data["y"] <= MAP_RECT[1] + MAP_RECT[3]))
print(f"share of valid gaze on the map panel: {on_map[gaze_data['valid']].mean():.0%}")

## Summary

| What we did | Tool |
|---|---|
| Loaded a real city as a street graph | `shasta.map.Map` |
| Described the simulation in a YAML file | `config.yaml`, `load_config` |
| Ordered a swarm through the Gym API, with no human | `ShastaEnv`, `env.step({group: node})` |
| Played the operator in the browser | `SwarmCommander`, `ShastaGUI`, `play_shasta` |
| Defined our own experiment and compared strategies | `BaseExperiment` subclass |
| Streamed orders and gaze with LSL and saved them as XDF | `pylsl`, `lsl_tools.Recorder` |
| Loaded the recording and measured gaze against orders | `pyxdf`, fixations, reaction time |

The same four steps fit any human-swarm study: **simulate, interact, record, analyse.**